# PocketHarvey: Preprocessing Pipeline for an Offline Indian Legal Corpus

This notebook documents the full preprocessing pipeline for building a specialized legal corpus intended to fine-tune the Bonsai-8B-mlx-1bit model. The goal is to produce a clean, deduplicated, quality-filtered dataset of Indian legal text that can be used for continued pretraining and supervised fine-tuning (SFT).

The pipeline is structured as follows:

1. Environment setup and dependency installation
2. Raw data acquisition from free, publicly licensed sources
3. Text normalization and language filtering (FineWeb-style heuristics)
4. Quality scoring and document filtering (FineWebEdu-inspired)
5. MinHash-based near-deduplication (Datatrove-style)
6. Perplexity-proxy scoring for curriculum tiering (SmolLM/Nemotron strategy)
7. Synthetic instruction pair generation (NeMo DataDesigner-inspired, no API)
8. Dataset mixing with staged curriculum ratios
9. Export to Parquet for downstream training

No paid APIs are used at any stage. All source datasets are either Apache 2.0 or CC-BY licensed.

**Runtime note:** This notebook is designed for Google Colab with a T4 GPU. Steps 1-6 run fine on CPU. Step 7 (synthetic generation) benefits from GPU.

## 1. Environment Setup

We install the following:

- `datatrove`: HuggingFace's large-scale text processing library, which is the backbone of the FineWeb pipeline. We use its pipeline abstractions, readers, and filters.
- `datasets`: Standard HuggingFace library for loading public datasets from the Hub.
- `datasketch`: Provides MinHash and LSH implementations for near-duplicate detection, mirroring the deduplication step in FineWeb.
- `trafilatura`: Web text extraction library used in FinePDF-style pipelines for cleaning raw HTML and PDF-extracted text.
- `ftfy`: Fixes unicode encoding issues common in legal text scraped from government websites.
- `langdetect`: Lightweight language detection. We use this to keep only English, Hindi, and Marathi documents.
- `transformers` + `sentencepiece`: For tokenization when computing token-length-based quality signals.
- `nltk`: Sentence tokenization for the synthetic generation step.

In [ ]:
!pip install -q \
    "datatrove[processing]" \
    datasets \
    datasketch \
    trafilatura \
    ftfy \
    langdetect \
    transformers \
    sentencepiece \
    nltk \
    pyarrow \
    pandas \
    tqdm \
    huggingface_hub

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

In [ ]:
import os
import re
import json
import math
import random
import hashlib
import unicodedata
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
from typing import List, Dict, Optional, Tuple
from collections import Counter, defaultdict
from tqdm.auto import tqdm

import ftfy
from nltk.tokenize import sent_tokenize, word_tokenize
from datasketch import MinHash, MinHashLSH
from langdetect import detect, LangDetectException
from datasets import load_dataset, Dataset

# Reproducibility
random.seed(42)
np.random.seed(42)

# Directory layout
for d in ["data/raw", "data/filtered", "data/deduplicated", "data/scored", "data/synthetic", "data/final"]:
    Path(d).mkdir(parents=True, exist_ok=True)

print("Setup complete.")

## 2. Raw Data Acquisition

### 2.1 Source Selection Rationale

The corpus needs to cover Indian legal domains with factual grounding: constitutional law, criminal law (IPC/CrPC), civil procedure, consumer protection, labour law, family law, and RTI. We use the following sources:

**Source A — `joelniklaus/indian_legal_dataset` (HuggingFace Hub)**  
This dataset contains Indian Supreme Court and High Court judgments along with several downstream tasks (case summarization, rhetorical role labeling, court judgment prediction). The raw judgment text is the most factually grounded source we have. License: MIT.

**Source B — `law-ai/ildc_multi` (HuggingFace Hub)**  
ILDC (Indian Legal Document Corpus) is a large corpus of Supreme Court cases with binary judgment prediction labels. We only use the text, not the labels. License: CC BY 4.0.

**Source C — `HuggingFaceFW/fineweb-edu` (sampled subset)**  
FineWebEdu is a web-crawl derived dataset filtered by an educational quality classifier (scores 0-5; we keep >= 3). We stream a sample and apply keyword filtering to retain only documents relevant to Indian law. This gives us educational, plain-language legal explanations that complement the dense judgment text.

**Source D — Manually authored seed texts**  
For domains underrepresented in the above (RTI procedure, police rights, tenant rights), we write accurate plain-language summaries directly from the text of public domain Indian statutes. These are short but high-signal documents for SFT.

### 2.2 What we are NOT using

CommonCrawl WARC/WET files: These require either Spark or the datatrove distributed executor, and the download size is prohibitive for a single Colab session. The FineWebEdu sample already represents the useful slice of CommonCrawl for our purposes — it has already been crawled, extracted, and quality-scored. Pulling raw WARC is redundant here.

IndicLegalBERT corpus: The raw pretraining data is not publicly released. Only the model weights are.

IndianKanoon: Does not have a bulk download. Scraping violates their ToS.

In [ ]:
# Source A: Indian Legal Dataset — case summarization split
# This split contains the full judgment text alongside
# reference summaries. We only use the judgment text.

raw_docs = []

print("Loading Source A: joelniklaus/indian_legal_dataset...")
try:
    ds_a = load_dataset(
        "joelniklaus/indian_legal_dataset",
        "case_summarization",
        split="train",
        trust_remote_code=True
    )
    count_a = 0
    for row in tqdm(ds_a, desc="Source A"):
        # Column name varies depending on the config version
        text = row.get("judgment", "") or row.get("document", "") or row.get("text", "")
        if isinstance(text, str) and len(text.strip()) > 300:
            raw_docs.append({
                "text": text.strip(),
                "source": "joelniklaus_ilegal_summarization",
                "domain": "court_judgment",
                "lang": "en"
            })
            count_a += 1
    print(f"  Source A: {count_a} documents loaded.")
except Exception as e:
    print(f"  Source A failed: {e}")
    print("  This is non-fatal. Continuing with remaining sources.")

In [ ]:
# Source B: ILDC Multi — Supreme Court judgments
# Contains ~35k documents. We load the train split only.
# The 'text' field contains the full case text.

print("Loading Source B: law-ai/ildc_multi...")
try:
    ds_b = load_dataset("law-ai/ildc_multi", split="train", trust_remote_code=True)
    count_b = 0
    for row in tqdm(ds_b, desc="Source B"):
        text = row.get("text", "")
        if isinstance(text, str) and len(text.strip()) > 300:
            raw_docs.append({
                "text": text.strip(),
                "source": "ildc_multi",
                "domain": "court_judgment",
                "lang": "en"
            })
            count_b += 1
    print(f"  Source B: {count_b} documents loaded.")
except Exception as e:
    print(f"  Source B failed: {e}")
    print("  Continuing.")

In [ ]:
# Source C: FineWebEdu — streaming sample filtered for Indian legal content
#
# FineWebEdu is a 1.3T token dataset filtered by an educational quality
# classifier. The 'score' field is 0-5; HuggingFace recommends >= 3 for
# high-quality educational text.
#
# We stream the 10BT sample and stop once we have collected enough
# relevant documents. The keyword filter is aggressive because most
# FineWebEdu content is not about Indian law.

INDIAN_LEGAL_KEYWORDS = [
    "indian penal code", "ipc section", "crpc", "code of criminal procedure",
    "constitution of india", "supreme court of india", "high court india",
    "first information report", "fir india", "rti act", "right to information",
    "consumer protection act india", "domestic violence act india",
    "section 498a", "nalsa", "legal aid india", "lok adalat",
    "rent control india", "labour law india", "fundamental rights india",
    "pil india", "public interest litigation india", "dowry prohibition",
    "indian contract act", "transfer of property act india",
    "motor vehicles act india", "negotiable instruments act"
]

def is_indian_legal(text: str) -> bool:
    t = text.lower()
    return any(kw in t for kw in INDIAN_LEGAL_KEYWORDS)

print("Loading Source C: FineWebEdu sample (streaming)...")
count_c = 0
MAX_FINEWEB_DOCS = 5000  # cap to avoid excessive streaming time
FINEWEB_QUALITY_THRESHOLD = 3.0

try:
    ds_c = load_dataset(
        "HuggingFaceFW/fineweb-edu",
        name="sample-10BT",
        split="train",
        streaming=True
    )
    scanned = 0
    for row in ds_c:
        scanned += 1
        if scanned % 10000 == 0:
            print(f"  Scanned {scanned} FineWebEdu rows, collected {count_c} legal docs so far...")
        if count_c >= MAX_FINEWEB_DOCS:
            break
        score = row.get("score", 0)
        if score < FINEWEB_QUALITY_THRESHOLD:
            continue
        text = row.get("text", "")
        if not isinstance(text, str) or len(text.strip()) < 300:
            continue
        if not is_indian_legal(text):
            continue
        raw_docs.append({
            "text": text.strip(),
            "source": "fineweb_edu",
            "domain": "web_educational",
            "lang": "en",
            "edu_score": score
        })
        count_c += 1
    print(f"  Source C: {count_c} relevant documents from {scanned} scanned.")
except Exception as e:
    print(f"  Source C failed: {e}")

In [ ]:
# Source D: Manually authored seed texts
#
# These are accurate plain-language summaries written directly from
# the text of public domain Indian statutes (IPC, CrPC, RTI Act,
# Consumer Protection Act 2019, Code on Wages 2019, PWDVA 2005,
# Model Tenancy Act 2021). They are short but high-signal.
#
# The purpose is to seed the corpus with the kind of plain-language
# explanatory text that we want the model to generate. Court judgments
# alone are too dense and formal for a public-facing legal literacy tool.

SEED_TEXTS = [
    {
        "text": """Right to Information Act, 2005 - Filing Procedure and Remedies

The Right to Information Act, 2005 (RTI Act) confers on every citizen the right to request information held by any public authority. The term 'public authority' covers central and state government departments, constitutional bodies, statutory corporations, local bodies such as gram panchayats and municipalities, and any entity that receives substantial government funding.

Filing an RTI application: The application must be addressed to the Public Information Officer (PIO) of the relevant department. It can be submitted by post, by hand, or online at rtionline.gov.in for Central Government departments. The prescribed fee is Rs. 10 for Central Government applications, paid by demand draft, postal order, or online payment. Applicants holding a valid BPL (Below Poverty Line) certificate are fully exempt from all fees under the Act.

Response timelines: The PIO must provide information within 30 days of receiving the application. If the request pertains to the life or liberty of any person, the deadline is 48 hours. Failure to respond within the prescribed period is treated as a deemed refusal.

Appellate remedies: If the PIO refuses the request, provides incomplete information, or does not respond, the applicant may file a First Appeal to the First Appellate Authority (FAA) within 30 days of the refusal or the expiry of the response period. If the First Appeal is also unsatisfactory, a Second Appeal or complaint may be filed with the Central Information Commission (CIC) or the relevant State Information Commission (SIC) within 90 days of the FAA order.

Exemptions under Section 8: Certain categories of information are exempt, including information that would prejudicially affect national security or sovereignty, cabinet papers, information received in confidence from foreign governments, information that would constitute contempt of court, and personal information that has no relationship to any public activity or interest.

This is general educational information based on the text of the RTI Act, 2005. For advice on a specific RTI matter, consult a qualified lawyer or contact the National Legal Services Authority (NALSA) helpline at 1800-110-001.""",
        "source": "seed_manual",
        "domain": "rti",
        "lang": "en"
    },
    {
        "text": """Consumer Protection Act, 2019 - Rights, Forums, and Complaint Procedure

The Consumer Protection Act, 2019 (CPA 2019) replaces the 1986 Act and extends its coverage to e-commerce transactions. A 'consumer' is defined as a person who buys goods or avails services for personal use and not for commercial resale. Business purchasers are excluded.

Six consumer rights recognized under Section 2(9): (1) Right to safety against hazardous goods and services. (2) Right to information about quality, quantity, potency, purity, standard, and price. (3) Right to be assured of access to goods and services at competitive prices. (4) Right to be heard before consumer-affecting decisions. (5) Right to seek redressal against unfair trade practices or restrictive trade practices. (6) Right to consumer education.

Forum jurisdiction by pecuniary value: District Consumer Disputes Redressal Commission handles claims up to Rs. 1 crore. State Consumer Disputes Redressal Commission handles claims between Rs. 1 crore and Rs. 10 crore. National Consumer Disputes Redressal Commission (NCDRC) handles claims above Rs. 10 crore. Online complaints for all forums can be filed at edaakhil.nic.in at no cost to the complainant.

Complaint procedure: The complainant files a written complaint before the appropriate forum. The opposite party (manufacturer, seller, or service provider) is issued notice and given an opportunity to respond. The forum may appoint an expert to examine defective goods. Orders can include replacement, repair, refund, compensation, and discontinuation of the unfair practice.

Mediation: CPA 2019 introduces a Consumer Mediation Cell at each district and state commission. Parties may be referred to mediation at any stage before final order. Mediation is voluntary and its settlement is binding only if both parties consent in writing.

This is general educational information. Consult a qualified consumer rights advocate or NALSA for specific case advice.""",
        "source": "seed_manual",
        "domain": "consumer_protection",
        "lang": "en"
    },
    {
        "text": """Wages and Payment Protections for Workers — Code on Wages, 2019

The Code on Wages, 2019 consolidates four earlier labour laws: the Payment of Wages Act 1936, the Minimum Wages Act 1948, the Payment of Bonus Act 1965, and the Equal Remuneration Act 1976. It extends the concept of minimum wage to all workers regardless of sector or wage ceiling, which was not the case under the earlier fragmented framework.

Minimum wage obligations: Every employer must pay at least the minimum rate of wages notified by the Central or State Government for the relevant scheduled employment. Wages must be paid without unauthorized deductions. The Government revises minimum wages at intervals not exceeding five years.

Payment timelines: Wages must be paid on a working day. For establishments employing fewer than 1000 workers, wages must be paid by the 7th of the following month. For larger establishments, by the 10th. Wages can be paid by cash, cheque, or direct bank transfer.

Prohibited deductions: Only deductions explicitly authorized under the Code are permitted. These include fines (subject to strict procedural requirements), absence from duty, housing and amenities provided by the employer, advances given, and income tax. Deductions for damage or loss require an opportunity for the worker to show cause.

Remedies for underpayment: A worker who believes wages are below the minimum may file a claim before the Authority appointed under the Code (typically the Labour Commissioner or an officer designated by the State Government). The Authority can award underpaid wages along with a compensatory penalty up to 10 times the underpaid amount.

MGNREGA rights: Under the Mahatma Gandhi National Rural Employment Guarantee Act, 2005, rural households are entitled to a minimum of 100 days of unskilled manual work per financial year. Wages must be paid within 15 days. Delay attracts a compensation payment to the worker.

This is general educational information. State-specific minimum wages, which vary by industry and skill category, are available from your district's Labour Department. For specific advice, consult your State Legal Services Authority.""",
        "source": "seed_manual",
        "domain": "labour_law",
        "lang": "en"
    },
    {
        "text": """Section 498A IPC and the Protection of Women from Domestic Violence Act, 2005

Section 498A of the Indian Penal Code (IPC): Inserted in 1983, Section 498A criminalizes cruelty by a husband or his relatives toward a married woman. 'Cruelty' includes: (a) any conduct that is likely to drive the woman to commit suicide or to cause grave injury or danger to her life, limb, or health, whether mental or physical; and (b) harassment of the woman or her relatives to coerce her or them to meet any unlawful demand for property or valuable security.

This offence is cognizable (police can arrest without a warrant), non-bailable (bail is not a right; a magistrate must decide), and non-compoundable (the parties cannot withdraw the complaint without court permission). Punishment is imprisonment up to three years and a fine.

Protection of Women from Domestic Violence Act, 2005 (PWDVA): The PWDVA provides civil remedies for domestic violence, complementing the criminal remedy under 498A. It covers physical, sexual, verbal, emotional, and economic abuse. Importantly, it extends protection to women in live-in relationships and to other female members of the shared household.

Remedies under PWDVA: Protection Orders (restraining the respondent from committing further acts of violence), Residence Orders (ensuring the woman's right to stay in the shared household), Monetary Relief (for medical expenses, loss of earnings, maintenance), and Custody Orders for children. Applications can be filed before the Magistrate's Court. Protection Officers appointed by the state government assist complainants in filing.

Dowry Prohibition Act, 1961: Giving, taking, or demanding dowry is punishable with imprisonment of not less than five years and a fine of not less than Rs. 15,000 or the value of the dowry, whichever is higher.

Emergency contacts: Women's Helpline: 181 (free, available 24x7 in most states). Police: 100. One Stop Centres (Sakhi Centres) provide shelter, medical assistance, police facilitation, and legal aid at a single location. NALSA provides free legal services: 1800-110-001.

This is general educational information only. Cases under 498A and PWDVA are highly fact-specific. Please consult a qualified advocate or your State Legal Services Authority.""",
        "source": "seed_manual",
        "domain": "criminal_family_law",
        "lang": "en"
    },
    {
        "text": """Tenant Rights in India — Model Tenancy Act, 2021 and State Rent Control Laws

Tenancy in India is regulated at the state level. Most states have their own Rent Control Acts, many based on the colonial-era framework. The Model Tenancy Act, 2021 is a central advisory framework that states may adopt in full or in part.

Key provisions of the Model Tenancy Act, 2021: All tenancy agreements must be in writing and submitted to a designated Rent Authority. Security deposits are capped at two months' equivalent rent for residential premises and six months for non-residential premises. Landlords cannot unilaterally increase rent during the tenancy agreement period. The landlord must provide 24-hour written notice before entering the property for inspections.

Essential services: A landlord who willfully cuts off electricity, water, or any other essential service to force the tenant to vacate commits an offence. The tenant can apply to the Rent Authority for restoration of services.

Eviction procedure: A landlord cannot evict a tenant without following due process. Valid grounds for eviction generally include: non-payment of rent beyond a specified period, subletting without permission, material damage to the property, and bona fide personal requirement by the landlord. The landlord must issue a written notice. If the tenant does not comply, the landlord must file a case before the Rent Court. A tenant cannot be forcibly evicted without a court order.

Dispute resolution: Disputes between landlords and tenants are adjudicated by the Rent Authority at the district level. Appeals lie to a Rent Court and further to a Rent Tribunal.

Practical steps for harassed tenants: (1) Preserve all communications with the landlord in writing. (2) Send a formal complaint to the Rent Authority. (3) If essential services are cut, file an urgent application before the Rent Authority or civil court. (4) For illegal eviction or forceful entry, file an FIR under Section 441 IPC (criminal trespass) or Section 503 IPC (criminal intimidation) as appropriate.

This is general educational information. State-specific rent control laws differ significantly. Consult a local advocate or your State Legal Services Authority for advice.""",
        "source": "seed_manual",
        "domain": "tenancy_law",
        "lang": "en"
    },
    {
        "text": """Rights During Police Arrest and FIR Filing — Constitutional and Statutory Protections

First Information Report (FIR): An FIR is the document prepared by police upon receiving information about a cognizable offence. A cognizable offence is one where police have the power to arrest without a warrant (e.g., theft, assault, rape, murder, cheating above a threshold). Any person — the victim, a witness, or any member of the public — can report a cognizable offence.

Police obligation to register FIR: Under Section 154 CrPC (now Section 173 BNSS, Bharatiya Nagarik Suraksha Sanhita, 2023), police are legally obligated to register an FIR for every cognizable offence. Refusal is illegal. Remedies if police refuse: (1) Send the complaint by registered post to the Superintendent of Police, who must either investigate or direct registration. (2) File a private complaint before a Judicial Magistrate under Section 156(3) CrPC, who can direct the police to register and investigate.

Rights of the arrested person under Article 22 of the Constitution: (1) Right to be informed of the grounds of arrest at the time of arrest. (2) Right to consult and be defended by a legal practitioner of choice. (3) Right to be produced before the nearest Magistrate within 24 hours of arrest, excluding travel time. (4) Right not to be detained beyond 24 hours without Magistrate authorization.

Additional statutory rights: (1) Right to free legal aid if the accused cannot afford a lawyer — the Magistrate is required to inform the accused of this right. (2) Right to silence under Article 20(3) — no person accused of an offence can be compelled to be a witness against themselves. (3) Women arrested on any charge cannot be arrested after sunset and before sunrise, except in exceptional circumstances and only in the presence of a female police officer. (4) The arresting officer must prepare a memo of arrest, attested by a witness and countersigned by the arrestee.

Prohibition on torture and custodial violence: Custodial violence by police is a cognizable offence. The Supreme Court in D.K. Basu v. State of West Bengal (1997) laid down detailed guidelines on arrest procedure that remain binding. Victims of custodial violence may file a complaint with the State Human Rights Commission or the National Human Rights Commission.

This is general educational information based on the Constitution of India and CrPC/BNSS. Emergency: dial 100. Free legal aid: NALSA helpline 1800-110-001.""",
        "source": "seed_manual",
        "domain": "criminal_procedure",
        "lang": "en"
    }
]

raw_docs.extend(SEED_TEXTS)
print(f"Source D: {len(SEED_TEXTS)} seed documents added.")
print(f"Total raw documents across all sources: {len(raw_docs)}")

## 3. Text Normalization and Basic Filtering

This step handles the lowest-level cleanup before we apply quality heuristics. It is analogous to the initial text extraction and normalization stage in the FineWeb pipeline.

**What we do here:**

1. **Unicode normalization**: Fix broken unicode using `ftfy`. Government PDFs and older court judgment digitizations often have encoding artifacts.
2. **Whitespace normalization**: Collapse excessive whitespace, remove null bytes, normalize line endings.
3. **Minimum length filter**: Drop documents under 200 characters. These are almost always navigation text or OCR fragments.
4. **Language detection**: Keep only documents detected as English (`en`), Hindi (`hi`), or Marathi (`mr`). We use `langdetect` with a fallback to `en` for texts where detection fails but script looks Latin.
5. **Script-based sanity check**: Reject documents that are predominantly numeric or symbolic with very little natural language content.

We do not use `trafilatura` for HTML extraction here because our sources are already text. `trafilatura` would be used in a CommonCrawl WARC processing pipeline (the `datatrove` pipeline with `WarcReader -> TrafilaturaTitleExtractor -> ...`). Since we skipped raw WARC for this session, we note where it would slot in.

In [ ]:
# Text normalization utilities
#
# In a full Datatrove pipeline, these would be implemented as
# PipelineStep subclasses. Here we implement them as standalone
# functions and apply them manually, which is equivalent for a
# single-machine run.

ALLOWED_LANGS = {"en", "hi", "mr"}

def normalize_text(text: str) -> str:
    """
    Apply unicode normalization and whitespace cleanup.
    ftfy.fix_text handles encoding artifacts like garbled unicode
    and incorrect byte sequences, which are common in court documents.
    """
    text = ftfy.fix_text(text)
    # Normalize unicode to NFC form (composed characters)
    text = unicodedata.normalize("NFC", text)
    # Remove null bytes and other control characters except newlines and tabs
    text = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]", "", text)
    # Collapse 3+ consecutive newlines to 2
    text = re.sub(r"\n{3,}", "\n\n", text)
    # Collapse excessive spaces
    text = re.sub(r"[ \t]{3,}", "  ", text)
    return text.strip()


def detect_language(text: str) -> Optional[str]:
    """
    Detect language using langdetect.
    langdetect is non-deterministic by default; we seed it for reproducibility.
    Returns None if detection fails or confidence is too low.
    """
    from langdetect import detect_langs
    try:
        # Use a snippet of the text for speed
        snippet = text[:2000]
        results = detect_langs(snippet)
        if results:
            top = results[0]
            # Only accept detection if confidence >= 0.7
            if top.prob >= 0.7:
                return top.lang
    except LangDetectException:
        pass
    return None


def is_predominantly_text(text: str, min_alpha_ratio: float = 0.6) -> bool:
    """
    Reject documents that are mostly numbers, punctuation, or symbols.
    A well-formed legal document should be at least 60% alphabetic characters.
    """
    if not text:
        return False
    alpha_count = sum(1 for c in text if c.isalpha())
    return (alpha_count / len(text)) >= min_alpha_ratio


print("Normalization functions defined.")

In [ ]:
# Apply normalization and basic filtering

MIN_CHAR_LENGTH = 200

normalized_docs = []
filter_stats = Counter()

for doc in tqdm(raw_docs, desc="Normalizing"):
    text = doc.get("text", "")
    if not isinstance(text, str):
        filter_stats["not_string"] += 1
        continue

    # Step 1: Normalize
    text = normalize_text(text)

    # Step 2: Minimum length
    if len(text) < MIN_CHAR_LENGTH:
        filter_stats["too_short"] += 1
        continue

    # Step 3: Must be mostly alphabetic
    if not is_predominantly_text(text, min_alpha_ratio=0.55):
        filter_stats["non_text"] += 1
        continue

    # Step 4: Language filter
    # For seed texts and known-English sources, trust the declared lang.
    # For web sources, detect independently.
    declared_lang = doc.get("lang", None)
    if declared_lang in ALLOWED_LANGS and doc.get("source") != "fineweb_edu":
        lang = declared_lang
    else:
        lang = detect_language(text)
        if lang is None:
            # Fallback: if text is mostly ASCII Latin, assume English
            ascii_ratio = sum(1 for c in text if ord(c) < 128) / len(text)
            lang = "en" if ascii_ratio > 0.85 else None
        if lang not in ALLOWED_LANGS:
            filter_stats["wrong_language"] += 1
            continue

    new_doc = dict(doc)
    new_doc["text"] = text
    new_doc["lang"] = lang
    new_doc["char_count"] = len(text)
    normalized_docs.append(new_doc)
    filter_stats["kept"] += 1

print(f"Normalization complete.")
print(f"Filter statistics: {dict(filter_stats)}")
print(f"Documents passing basic filter: {len(normalized_docs)} / {len(raw_docs)}")

## 4. FineWeb-Style Quality Heuristics

FineWeb and FineWebEdu apply a battery of rule-based heuristics to filter out low-quality text before running the more expensive classifier. These heuristics were developed iteratively by examining what kinds of text a character-level n-gram or ML classifier tends to reject, and codifying them as deterministic rules.

We adapt the most relevant FineWeb heuristics for a legal corpus:

**Line-level heuristics:**
- `stop_word_ratio`: The fraction of lines that end with a stop word (common in real prose; absence suggests navigation menus or table of contents fragments).
- `bullet_ratio`: If more than 90% of lines start with a bullet character, the document is probably a scraped table or list, not substantive text.
- `short_line_ratio`: A very high fraction of short lines (< 30 characters) indicates OCR artifacts or header/footer spam.

**Character-level heuristics:**
- `symbol_to_word_ratio`: High ratio of `#`, `|`, `{`, etc. to words indicates a code dump or wiki markup artifact.
- `ellipsis_ratio`: Excessive ellipsis (`...`) indicates truncated or low-quality web snippets.
- `digit_ratio`: Ratio of digit characters to total characters. Legal text has numbers but predominantly text.

**Word-level heuristics:**
- `type_token_ratio` (TTR): Vocabulary diversity. Very low TTR means the document is highly repetitive (e.g., duplicate paragraphs, templated filler).
- `unique_word_ratio`: Similar to TTR but computed over the full document.

**Document-level heuristics:**
- Presence of at least one legal anchor term (statute name, section reference, court name) — ensures the document is actually about law and not tangentially related.

Each heuristic produces a boolean pass/fail or a score. Documents failing any hard threshold are dropped. We log reasons for analysis.

In [ ]:
# FineWeb-style quality heuristics implementation

LEGAL_ANCHOR_TERMS = [
    r"section \d+",
    r"article \d+",
    r"ipc", r"crpc", r"bnss", r"bns",
    r"supreme court", r"high court", r"district court",
    r"magistrate", r"advocate", r"plaintiff", r"defendant",
    r"petitioner", r"respondent", r"judgment", r"verdict",
    r"affidavit", r"writ petition", r"habeas corpus",
    r"consumer forum", r"labour commissioner", r"rti",
    r"fir", r"chargesheet", r"bail", r"cognizable",
    r"constitution of india", r"fundamental rights",
    r"legal aid", r"nalsa", r"lok adalat"
]

LEGAL_ANCHOR_RE = re.compile(
    "|".join(LEGAL_ANCHOR_TERMS),
    re.IGNORECASE
)

BULLET_CHARS = set("•*-–—►▶●○◦")


def compute_quality_signals(text: str) -> Dict[str, float]:
    """
    Compute the full set of FineWeb-style quality signals for a document.
    Returns a dictionary of signal_name -> value.
    """
    signals = {}
    lines = text.split("\n")
    non_empty_lines = [l for l in lines if l.strip()]
    total_lines = max(len(non_empty_lines), 1)

    # --- Line-level signals ---

    # Fraction of lines that are very short (< 30 chars)
    short_lines = sum(1 for l in non_empty_lines if len(l.strip()) < 30)
    signals["short_line_ratio"] = short_lines / total_lines

    # Fraction of lines starting with a bullet character
    bullet_lines = sum(1 for l in non_empty_lines if l.strip() and l.strip()[0] in BULLET_CHARS)
    signals["bullet_ratio"] = bullet_lines / total_lines

    # --- Character-level signals ---

    total_chars = max(len(text), 1)

    # Symbol to character ratio (excludes standard punctuation)
    symbol_chars = sum(1 for c in text if c in "#|{}[]<>^~`\\")
    signals["symbol_ratio"] = symbol_chars / total_chars

    # Digit to character ratio
    digit_chars = sum(1 for c in text if c.isdigit())
    signals["digit_ratio"] = digit_chars / total_chars

    # Ellipsis ratio
    ellipsis_count = text.count("...") + text.count("…")
    word_count = max(len(text.split()), 1)
    signals["ellipsis_ratio"] = ellipsis_count / word_count

    # --- Word-level signals ---

    words = text.lower().split()
    word_count = max(len(words), 1)
    unique_words = set(words)

    # Type-token ratio on first 500 words to normalize for document length
    sample = words[:500]
    signals["type_token_ratio"] = len(set(sample)) / max(len(sample), 1)

    # Unique word ratio over full document
    signals["unique_word_ratio"] = len(unique_words) / word_count

    # --- Document-level signals ---

    # Does the document contain at least one legal anchor term?
    signals["has_legal_anchor"] = 1.0 if LEGAL_ANCHOR_RE.search(text) else 0.0

    return signals


def passes_quality_heuristics(signals: Dict[str, float]) -> Tuple[bool, str]:
    """
    Apply hard thresholds to quality signals.
    Returns (passed, reason_for_rejection).
    Thresholds are based on FineWeb's published heuristics, adjusted
    for legal text which naturally has higher digit ratios and more
    structured formatting than general web text.
    """
    if signals["short_line_ratio"] > 0.70:
        return False, "short_line_ratio"
    if signals["bullet_ratio"] > 0.85:
        return False, "bullet_ratio"
    if signals["symbol_ratio"] > 0.05:
        return False, "symbol_ratio"
    if signals["digit_ratio"] > 0.20:
        return False, "digit_ratio"
    if signals["ellipsis_ratio"] > 0.10:
        return False, "ellipsis_ratio"
    if signals["type_token_ratio"] < 0.10:
        return False, "low_type_token_ratio"
    if signals["has_legal_anchor"] == 0.0:
        return False, "no_legal_anchor"
    return True, ""


print("Quality heuristic functions defined.")

In [ ]:
# Apply quality heuristics to all normalized documents

quality_filtered = []
quality_rejection_reasons = Counter()

for doc in tqdm(normalized_docs, desc="Quality filtering"):
    signals = compute_quality_signals(doc["text"])
    passed, reason = passes_quality_heuristics(signals)

    if not passed:
        quality_rejection_reasons[reason] += 1
        continue

    doc["quality_signals"] = signals
    quality_filtered.append(doc)

print(f"Quality filtering complete.")
print(f"Documents remaining: {len(quality_filtered)} / {len(normalized_docs)}")
print(f"Rejection reasons: {dict(quality_rejection_reasons)}")

In [ ]:
# Save the quality-filtered intermediate result

df_filtered = pd.DataFrame([
    {
        "text": d["text"],
        "source": d["source"],
        "domain": d["domain"],
        "lang": d["lang"],
        "char_count": d["char_count"],
        "type_token_ratio": d["quality_signals"]["type_token_ratio"],
        "digit_ratio": d["quality_signals"]["digit_ratio"],
    }
    for d in quality_filtered
])

df_filtered.to_parquet("data/filtered/quality_filtered.parquet", index=False)
print(f"Saved {len(df_filtered)} documents to data/filtered/quality_filtered.parquet")
print(df_filtered[["source", "domain", "lang", "char_count"]].describe())

## 5. Near-Duplicate Removal — MinHash LSH

Duplication is one of the most damaging properties in a pretraining corpus. A model trained on a corpus where the same document appears many times will overfit to those documents, degrading generalization. This is especially relevant for legal corpora, where the same judgment text may appear in multiple datasets or the same legal provision may be copy-pasted across dozens of websites.

FineWeb uses MinHash Locality-Sensitive Hashing (LSH) for near-duplicate detection — this is the same approach as the datatrove `MinhashDeduplanner` step. The algorithm is as follows:

1. Tokenize each document into character n-grams (we use 5-grams, as in FineWeb).
2. Compute a MinHash signature of 128 hash functions for each document's n-gram set.
3. Index all signatures into an LSH structure with a Jaccard similarity threshold (we use 0.7, meaning documents that share >70% of their n-grams are considered near-duplicates).
4. For each near-duplicate cluster, keep one representative document and discard the rest.

The datatrove `MinhashDeduplaner` does this in a distributed MapReduce fashion for billion-document corpora. For our scale (~tens of thousands of documents), we run it in-process.

We also apply exact deduplication first (SHA-256 hash of the normalized text), which is cheaper and catches the easiest duplicates.

In [ ]:
# Step 1: Exact deduplication via SHA-256
# Two documents with identical normalized text get identical hashes.
# We keep the first occurrence.

seen_hashes = set()
exact_deduped = []

for doc in quality_filtered:
    doc_hash = hashlib.sha256(doc["text"].encode("utf-8")).hexdigest()
    if doc_hash not in seen_hashes:
        seen_hashes.add(doc_hash)
        doc["doc_hash"] = doc_hash
        exact_deduped.append(doc)

print(f"Exact deduplication: {len(quality_filtered)} -> {len(exact_deduped)} documents")
print(f"Removed: {len(quality_filtered) - len(exact_deduped)} exact duplicates")

In [ ]:
# Step 2: Near-duplicate removal via MinHash LSH
#
# Parameters:
#   num_perm=128: Number of hash permutations for the MinHash signature.
#       More permutations -> more accurate Jaccard estimation but slower.
#       128 is the standard used in FineWeb.
#   threshold=0.7: Jaccard similarity threshold. Two documents with
#       Jaccard(n-gram sets) >= 0.7 are considered near-duplicates.
#       FineWeb uses 0.7. Lower values remove more aggressively.
#   ngram_size=5: Character 5-grams. This is standard for deduplication;
#       it captures local text reuse without being too sensitive to
#       minor rephrasing.

NUM_PERM = 128
JACCARD_THRESHOLD = 0.7
NGRAM_SIZE = 5


def get_char_ngrams(text: str, n: int) -> List[str]:
    """Extract character n-grams from text."""
    text = text.lower()
    # Remove whitespace normalization artifacts for stable n-gram computation
    text = re.sub(r"\s+", " ", text)
    return [text[i:i+n] for i in range(len(text) - n + 1)]


def compute_minhash(text: str, num_perm: int, ngram_size: int) -> MinHash:
    """Compute a MinHash signature for a document."""
    m = MinHash(num_perm=num_perm)
    ngrams = get_char_ngrams(text, ngram_size)
    # Use first 50000 n-grams maximum to bound memory and time
    # for very long court judgments
    for ng in ngrams[:50000]:
        m.update(ng.encode("utf-8"))
    return m


print("Computing MinHash signatures...")
print(f"Parameters: num_perm={NUM_PERM}, threshold={JACCARD_THRESHOLD}, ngram_size={NGRAM_SIZE}")

signatures = []
for doc in tqdm(exact_deduped, desc="MinHash"):
    mh = compute_minhash(doc["text"], NUM_PERM, NGRAM_SIZE)
    signatures.append(mh)

print(f"Signatures computed for {len(signatures)} documents.")

In [ ]:
# Build the LSH index and identify near-duplicate clusters
#
# We use a greedy approach: iterate through documents in order.
# For each document, query the LSH index. If no near-duplicate is found,
# add it to the index and mark it as kept. If a near-duplicate is found,
# discard the current document (the one already in the index is the
# representative).
#
# This is equivalent to the single-pass deduplication in datatrove's
# MinhashDeduplaner when run on a single machine.

lsh = MinHashLSH(threshold=JACCARD_THRESHOLD, num_perm=NUM_PERM)

kept_indices = []
near_dup_count = 0

for idx, (doc, mh) in enumerate(tqdm(zip(exact_deduped, signatures), total=len(exact_deduped), desc="LSH dedup")):
    key = doc["doc_hash"]
    try:
        result = lsh.query(mh)
        if result:
            # A near-duplicate already exists in the index
            near_dup_count += 1
        else:
            # New unique document: insert into index
            lsh.insert(key, mh)
            kept_indices.append(idx)
    except Exception:
        # If a key collision occurs (exact dup that slipped through), skip
        near_dup_count += 1

near_deduped = [exact_deduped[i] for i in kept_indices]

print(f"Near-duplicate removal complete.")
print(f"Before: {len(exact_deduped)} | After: {len(near_deduped)} | Removed: {near_dup_count}")

In [ ]:
# Save deduplicated checkpoint

df_deduped = pd.DataFrame([
    {
        "text": d["text"],
        "source": d["source"],
        "domain": d["domain"],
        "lang": d["lang"],
        "char_count": d["char_count"],
    }
    for d in near_deduped
])

df_deduped.to_parquet("data/deduplicated/deduped_corpus.parquet", index=False)
print(f"Saved {len(df_deduped)} deduplicated documents.")
print("Source distribution after deduplication:")
print(df_deduped["source"].value_counts())

## 6. Quality Scoring and Curriculum Tiering

### 6.1 Motivation

Not all documents that survive deduplication are equally valuable for training. A Supreme Court judgment on a property dispute from 1987 written in dense legalese is factually accurate but hard for a language model to generalize plain-language explanations from. An educational web page explaining RTI filing in simple English is more directly useful for the task we care about.

This is the core insight behind both FineWebEdu (use an educational quality classifier to weight documents) and the SmolLM/Nemotron data strategy (curate multiple quality tiers and mix them deliberately in a staged curriculum).

### 6.2 Scoring Approach

A full FineWebEdu-style scorer requires a trained classifier. We have two options:

**Option A (ideal):** Load the Bonsai-8B-mlx-1bit model (or a small proxy model like `HuggingFaceTB/SmolLM-360M`) and compute perplexity of each document on a reference educational legal text. Lower perplexity = more similar to the reference distribution = higher quality for our use case. This is the Bonsai-as-scorer idea from the proposal.

**Option B (what we implement here):** Since loading a full LLM in Colab just for scoring is wasteful, we use a lightweight composite score based on:
- Lexical diversity (type-token ratio)
- Average sentence length (too short = telegraphic; too long = unreadable)
- Presence of explanatory vocabulary (words like "means", "defined", "under section", "right", "shall", "penalty", etc.)
- Plain-language ratio (fraction of words from a common English vocabulary list)
- Structural score (presence of multiple paragraphs, reasonable length)

This composite score is a proxy for educational quality. It is not as accurate as a trained classifier, but it is deterministic, fast, and free.

### 6.3 Curriculum Tiers (SmolLM/Nemotron Strategy)

Based on the composite score, we assign each document to one of three tiers:

- **Tier 1 (high quality):** Dense, educational, plain-language legal explanations. Directly used for SFT.
- **Tier 2 (medium quality):** Court judgments with moderate readability. Used for continued pretraining.
- **Tier 3 (lower quality):** Dense legal text or procedural documents with little explanatory content. Used in smaller proportion during early pretraining.

Final mixing ratios (SmolLM-inspired staged curriculum):
- Tier 1: 50% of final training data
- Tier 2: 35%
- Tier 3: 15%

In [ ]:
# Quality scoring: composite educational score

# Common English words list proxy — top 3000 most frequent English words
# We approximate this by checking against a small hardcoded set of
# function words and common content words.
# A production implementation would use wordfreq or a full frequency list.
COMMON_WORDS = set([
    "the", "a", "an", "is", "are", "was", "were", "be", "been", "being",
    "have", "has", "had", "do", "does", "did", "will", "would", "could",
    "should", "may", "might", "shall", "can", "not", "and", "or", "but",
    "if", "in", "on", "at", "to", "for", "of", "with", "by", "from",
    "this", "that", "these", "those", "it", "its", "he", "she", "they",
    "you", "we", "who", "which", "what", "when", "where", "how", "why",
    "any", "all", "each", "every", "some", "no", "as", "so", "than",
    "such", "under", "above", "also", "their", "there", "than", "then",
    "into", "out", "up", "down", "about", "after", "before", "between"
])

EXPLANATORY_VOCAB = set([
    "means", "defined", "definition", "refers", "includes", "excludes",
    "right", "rights", "obligation", "duty", "liable", "liable",
    "penalty", "punishable", "offence", "offense", "complaint",
    "procedure", "process", "steps", "apply", "application",
    "appeal", "authority", "commission", "court", "tribunal",
    "section", "act", "rule", "regulation", "order", "notice",
    "compensation", "remedy", "relief", "protection", "violation",
    "citizen", "worker", "employee", "employer", "tenant", "landlord",
    "consumer", "complainant", "accused", "police", "magistrate"
])


def compute_educational_score(text: str) -> float:
    """
    Compute a composite educational quality score in [0, 1].
    Higher = more suitable as direct training signal for a legal literacy model.
    """
    score_components = []

    words = text.lower().split()
    word_count = max(len(words), 1)

    # Component 1: Type-token ratio on first 300 words (vocabulary diversity)
    sample = words[:300]
    ttr = len(set(sample)) / max(len(sample), 1)
    # Good legal text: 0.4-0.7. Normalize to [0,1]
    score_components.append(min(ttr / 0.6, 1.0))

    # Component 2: Explanatory vocabulary density
    expl_count = sum(1 for w in words if w in EXPLANATORY_VOCAB)
    expl_density = expl_count / word_count
    # Target: ~3-8% of words are explanatory. Normalize.
    score_components.append(min(expl_density / 0.05, 1.0))

    # Component 3: Average sentence length
    try:
        sentences = sent_tokenize(text[:5000])  # limit for speed
        if sentences:
            avg_sent_len = np.mean([len(s.split()) for s in sentences])
            # Ideal range 15-35 words per sentence for legal educational text
            # Outside this range, score degrades
            if 15 <= avg_sent_len <= 35:
                sent_score = 1.0
            elif avg_sent_len < 15:
                sent_score = avg_sent_len / 15
            else:
                sent_score = max(0, 1 - (avg_sent_len - 35) / 50)
            score_components.append(sent_score)
    except Exception:
        pass

    # Component 4: Plain language ratio (common words)
    common_count = sum(1 for w in words if w in COMMON_WORDS)
    plain_ratio = common_count / word_count
    # Target: 30-50% of words are common/function words
    score_components.append(min(plain_ratio / 0.35, 1.0))

    # Component 5: Document length score
    # Ideal: 500-5000 words. Very long judgments score lower (too dense for SFT)
    if 500 <= word_count <= 5000:
        len_score = 1.0
    elif word_count < 500:
        len_score = word_count / 500
    else:
        len_score = max(0.3, 1 - (word_count - 5000) / 20000)
    score_components.append(len_score)

    if not score_components:
        return 0.0
    return float(np.mean(score_components))


print("Educational scoring function defined.")

In [ ]:
# Apply scoring and assign curriculum tiers

TIER1_THRESHOLD = 0.65  # High quality — direct SFT signal
TIER2_THRESHOLD = 0.40  # Medium quality — pretraining
# Below TIER2_THRESHOLD -> Tier 3 (lower quality pretraining)

scored_docs = []

for doc in tqdm(near_deduped, desc="Scoring"):
    edu_score = compute_educational_score(doc["text"])
    doc["edu_score"] = edu_score

    if edu_score >= TIER1_THRESHOLD:
        doc["tier"] = 1
    elif edu_score >= TIER2_THRESHOLD:
        doc["tier"] = 2
    else:
        doc["tier"] = 3

    scored_docs.append(doc)

tier_counts = Counter(d["tier"] for d in scored_docs)
print(f"Scoring complete.")
print(f"Tier distribution: {dict(tier_counts)}")
scores = [d["edu_score"] for d in scored_docs]
print(f"Score stats: mean={np.mean(scores):.3f}, std={np.std(scores):.3f}, "
      f"min={np.min(scores):.3f}, max={np.max(scores):.3f}")

## 7. Synthetic Instruction Pair Generation (NeMo DataDesigner-Inspired)

### 7.1 Why Synthetic Data

The corpus we have built so far is good for continued pretraining (the model learns the legal domain distribution). But for supervised fine-tuning (SFT), we need instruction-response pairs: a user asks a question, the model gives a helpful, safe, disclaimered answer.

There are very few free, labeled Indian legal Q&A datasets. NeMo DataDesigner (and similar tools like Magpie, Evol-Instruct, and the Nemotron-4 data recipe) address this by generating synthetic instruction pairs seeded from high-quality source documents.

### 7.2 Our Approach (Zero-Cost)

We implement a rule-based synthetic generation pipeline that:

1. Takes a Tier 1 document as the seed.
2. Extracts key information units: section references, rights, obligations, procedures, penalties.
3. Generates plausible user questions using question templates.
4. Generates answers by extracting the relevant text span and appending a standard disclaimer.

This is not as flexible as LLM-based generation (NeMo DataDesigner uses a teacher model to generate diverse instructions). However, it is:
- Free and offline
- Deterministic and auditable
- Factually grounded (answers are extracted verbatim from verified source documents)

A production pipeline would layer LLM-generated paraphrases on top of these template-based pairs, as described in the Nemotron-4 data recipe.

### 7.3 Question Templates

We define templates for several question types that reflect real queries a user of PocketHarvey would ask:
- Rights questions: "What are my rights if ...?"
- Procedure questions: "How do I file a complaint about ...?"
- Definition questions: "What does X mean under Indian law?"
- Scenario questions: "My employer has not paid my salary for 2 months. What can I do?"
- Penalty questions: "What is the punishment for X under Indian law?"

In [ ]:
# Synthetic instruction pair generation

DISCLAIMER = (
    "\n\nNote: This is general educational information based on publicly available Indian law. "
    "Laws may change and individual circumstances vary. "
    "For advice specific to your situation, consult a qualified lawyer or contact the "
    "National Legal Services Authority (NALSA) at 1800-110-001 (toll-free)."
)

# Question templates parameterized by domain
QUESTION_TEMPLATES = {
    "rti": [
        "How do I file an RTI application in India?",
        "What is the time limit for a public authority to respond to an RTI?",
        "What can I do if my RTI application is rejected?",
        "Who can file an RTI application in India?",
        "Are there any fees for filing an RTI application?",
        "What information is exempt from RTI disclosure?",
    ],
    "consumer_protection": [
        "Where can I file a consumer complaint in India?",
        "What are my rights as a consumer under Indian law?",
        "I bought a defective product. What can I do?",
        "How do I file an online consumer complaint in India?",
        "What is the maximum compensation I can get from a consumer forum?",
        "A service provider cheated me. What are my legal options?",
    ],
    "labour_law": [
        "My employer has not paid my salary for two months. What are my rights?",
        "What is the minimum wage in India?",
        "Can my employer deduct money from my salary without my consent?",
        "I was wrongfully terminated. What can I do under Indian law?",
        "What is MGNREGA and who is eligible?",
        "Where do I file a complaint if my employer violates wage laws?",
    ],
    "criminal_family_law": [
        "What is Section 498A of the IPC?",
        "What protection does Indian law give to women facing domestic violence?",
        "What is the difference between Section 498A IPC and the Domestic Violence Act?",
        "What should I do if I am facing domestic violence?",
        "Is giving dowry a crime in India?",
        "What are the remedies available under the Protection of Women from Domestic Violence Act?",
    ],
    "tenancy_law": [
        "Can my landlord evict me without notice in India?",
        "My landlord has cut off electricity to force me to leave. Is this legal?",
        "What is the maximum security deposit a landlord can charge in India?",
        "What are my rights as a tenant under Indian law?",
        "How do I file a complaint against a landlord in India?",
        "What is the Model Tenancy Act 2021?",
    ],
    "criminal_procedure": [
        "What is an FIR and how do I file one?",
        "Can the police refuse to file an FIR?",
        "What are my rights if I am arrested by police in India?",
        "Can I be detained by police for more than 24 hours without appearing before a magistrate?",
        "Am I entitled to a free lawyer if I cannot afford one in India?",
        "What should I do if the police use violence during interrogation?",
    ],
    "court_judgment": [
        "What legal principles does this court judgment establish?",
        "What was the core dispute in this case?",
        "What did the court decide and on what grounds?",
        "What rights of the parties were discussed in this judgment?",
        "What statute or provision did the court interpret in this case?",
    ],
    "web_educational": [
        "Can you explain this concept in simple terms?",
        "What are the key points from this legal explanation?",
        "What should a person know about this area of Indian law?",
        "Summarize the most important rights and procedures described here.",
    ]
}


def extract_answer_span(text: str, max_chars: int = 1500) -> str:
    """
    Extract a useful answer span from a source document.
    We take the first substantive portion of the text up to max_chars,
    trying to end at a sentence boundary.
    """
    if len(text) <= max_chars:
        return text
    # Try to find a sentence boundary near max_chars
    truncated = text[:max_chars]
    # Find last period, ?, or ! before the cutoff
    for punct in [". ", "? ", "! ", ".\n"]:
        last_idx = truncated.rfind(punct)
        if last_idx > max_chars * 0.5:
            return truncated[:last_idx + 1].strip()
    return truncated.strip()


def generate_pairs_from_doc(doc: Dict, num_pairs: int = 2) -> List[Dict]:
    """
    Generate instruction-response pairs from a single seed document.
    """
    domain = doc.get("domain", "web_educational")
    templates = QUESTION_TEMPLATES.get(domain, QUESTION_TEMPLATES["web_educational"])

    # Sample questions without replacement
    chosen_questions = random.sample(templates, min(num_pairs, len(templates)))
    answer_span = extract_answer_span(doc["text"])

    pairs = []
    for question in chosen_questions:
        pairs.append({
            "instruction": question,
            "input": "",
            "output": answer_span + DISCLAIMER,
            "source_doc_hash": doc.get("doc_hash", ""),
            "domain": domain,
            "source": "synthetic_template",
            "seed_edu_score": doc.get("edu_score", 0.0)
        })
    return pairs


print("Synthetic generation functions defined.")

In [ ]:
# Generate synthetic pairs from all Tier 1 documents
# and a sample of Tier 2 documents

tier1_docs = [d for d in scored_docs if d["tier"] == 1]
tier2_docs = [d for d in scored_docs if d["tier"] == 2]

# Use all Tier 1 docs for synthetic generation
# Sample up to 500 Tier 2 docs
tier2_sample = random.sample(tier2_docs, min(500, len(tier2_docs)))

synthetic_pairs = []

print(f"Generating synthetic pairs from {len(tier1_docs)} Tier 1 docs (2 pairs each)...")
for doc in tqdm(tier1_docs, desc="Tier 1 synthetic"):
    pairs = generate_pairs_from_doc(doc, num_pairs=2)
    synthetic_pairs.extend(pairs)

print(f"Generating synthetic pairs from {len(tier2_sample)} Tier 2 docs (1 pair each)...")
for doc in tqdm(tier2_sample, desc="Tier 2 synthetic"):
    pairs = generate_pairs_from_doc(doc, num_pairs=1)
    synthetic_pairs.extend(pairs)

df_synthetic = pd.DataFrame(synthetic_pairs)
df_synthetic.to_parquet("data/synthetic/synthetic_pairs.parquet", index=False)

print(f"\nGenerated {len(synthetic_pairs)} synthetic instruction pairs.")
print("Domain distribution:")
print(df_synthetic["domain"].value_counts())

## 8. Dataset Mixing — SmolLM/Nemotron Staged Curriculum

The SmolLM paper and the Nemotron-4 data recipe both emphasize that the *mixture* of data types matters as much as the quality of individual documents. The key findings are:

1. Starting pretraining with lower-quality but high-volume data allows the model to develop broad language understanding.
2. Gradually increasing the proportion of high-quality, domain-specific data guides the model toward the target distribution.
3. SFT data (instruction pairs) should be kept separate from pretraining data and used only in the SFT phase.

For our corpus, we implement the following mixing strategy:

**Pretraining mix (for continued pretraining of Bonsai-8B):**
- Tier 3 documents: 15% (raw court judgments, dense procedural text)
- Tier 2 documents: 35% (moderately readable judgments and educational text)
- Tier 1 documents: 50% (high-quality educational legal explanations)

**SFT mix:**
- All synthetic instruction pairs (100% — these are already high-quality by construction)

We export both as separate Parquet files with a unified schema:
- `text`: The full text (for pretraining) or formatted instruction (for SFT)
- `source`: Original data source
- `domain`: Legal domain
- `tier`: Quality tier (1/2/3 for pretraining, 0 for SFT)
- `split`: `pretrain` or `sft`

The SFT data is formatted in the Alpaca instruction format, which is compatible with most fine-tuning frameworks (LLamaFactory, Axolotl, etc.).

In [ ]:
# Build the pretraining split with controlled mixing ratios

TARGET_PRETRAIN_SIZE = min(50000, len(scored_docs))  # cap for Colab memory

# Target counts per tier
n_tier1 = int(TARGET_PRETRAIN_SIZE * 0.50)
n_tier2 = int(TARGET_PRETRAIN_SIZE * 0.35)
n_tier3 = int(TARGET_PRETRAIN_SIZE * 0.15)

tier1_pool = [d for d in scored_docs if d["tier"] == 1]
tier2_pool = [d for d in scored_docs if d["tier"] == 2]
tier3_pool = [d for d in scored_docs if d["tier"] == 3]

def sample_with_replacement_if_needed(pool, n):
    """Sample n items from pool. If pool is smaller than n, sample with replacement."""
    if len(pool) == 0:
        return []
    if len(pool) >= n:
        return random.sample(pool, n)
    else:
        # Sample with replacement, but flag this for the user
        print(f"  Warning: pool size {len(pool)} < target {n}. Sampling with replacement.")
        return random.choices(pool, k=n)

pretrain_docs = []

sampled_t1 = sample_with_replacement_if_needed(tier1_pool, n_tier1)
sampled_t2 = sample_with_replacement_if_needed(tier2_pool, n_tier2)
sampled_t3 = sample_with_replacement_if_needed(tier3_pool, n_tier3)

for docs, tier in [(sampled_t1, 1), (sampled_t2, 2), (sampled_t3, 3)]:
    for d in docs:
        pretrain_docs.append({
            "text": d["text"],
            "source": d["source"],
            "domain": d["domain"],
            "lang": d["lang"],
            "tier": tier,
            "edu_score": d["edu_score"],
            "split": "pretrain"
        })

# Shuffle the pretraining set so that tiers are interleaved
# This prevents gradient shocks from tier transitions during training
random.shuffle(pretrain_docs)

df_pretrain = pd.DataFrame(pretrain_docs)
print(f"Pretraining split: {len(df_pretrain)} documents")
print(f"Tier distribution: {df_pretrain['tier'].value_counts().to_dict()}")
print(f"Source distribution: {df_pretrain['source'].value_counts().to_dict()}")

In [ ]:
# Build the SFT split in Alpaca format
#
# Alpaca format:
# {
#   "instruction": "What are my rights if my employer doesn't pay salary?",
#   "input": "",
#   "output": "Under the Code on Wages, 2019...<disclaimer>"
# }
#
# Many fine-tuning frameworks also accept a 'text' field with the
# full prompt formatted as a single string. We provide both.

ALPACA_PROMPT_TEMPLATE = (
    "Below is a question about Indian law. "
    "Provide a clear, accurate, and helpful answer based on general legal knowledge.\n\n"
    "### Question:\n{instruction}\n\n"
    "### Answer:\n{output}"
)

sft_rows = []
for pair in synthetic_pairs:
    formatted = ALPACA_PROMPT_TEMPLATE.format(
        instruction=pair["instruction"],
        output=pair["output"]
    )
    sft_rows.append({
        "instruction": pair["instruction"],
        "input": pair["input"],
        "output": pair["output"],
        "text": formatted,
        "domain": pair["domain"],
        "source": pair["source"],
        "tier": 0,
        "split": "sft"
    })

df_sft = pd.DataFrame(sft_rows)
print(f"SFT split: {len(df_sft)} instruction pairs")
print(f"Domain distribution: {df_sft['domain'].value_counts().to_dict()}")

## 9. Final Validation and Export

Before writing the final Parquet files, we run a validation pass to catch any remaining data quality issues:

1. **Null checks**: Ensure no null values in critical columns (`text`, `source`, `domain`, `split`).
2. **Token length distribution**: Estimate token counts using a simple word-based proxy (1 word ≈ 1.3 tokens for English). Flag documents that exceed 4096 tokens — they will need to be chunked before training with most model configurations.
3. **Disclaimer presence**: For SFT output fields, verify that the NALSA disclaimer is present in every response.
4. **Duplicate text in SFT**: Run exact hash deduplication on the `output` field to prevent multiple identical answer spans.

We then export:
- `data/final/pretrain_corpus.parquet` — the pretraining split
- `data/final/sft_corpus.parquet` — the SFT split
- `data/final/dataset_card.json` — metadata for reproducibility

The Parquet format is the standard for HuggingFace dataset uploads and is compatible with `datasets.load_dataset("parquet", data_files=...)` for direct use in training scripts.

In [ ]:
# Validation pass

def validate_dataframe(df: pd.DataFrame, split_name: str, critical_cols: List[str]):
    print(f"\nValidating {split_name} split ({len(df)} rows)...")

    # Null check
    for col in critical_cols:
        null_count = df[col].isnull().sum()
        if null_count > 0:
            print(f"  ERROR: {null_count} null values in column '{col}'")
        else:
            print(f"  OK: '{col}' has no null values.")

    # Empty string check
    for col in ["text"] if "text" in df.columns else []:
        empty_count = (df[col].str.strip() == "").sum()
        if empty_count > 0:
            print(f"  WARNING: {empty_count} empty strings in column '{col}'")

    # Token length estimation (word-based proxy)
    text_col = "text"
    if text_col in df.columns:
        word_counts = df[text_col].str.split().apply(len)
        est_tokens = word_counts * 1.3  # rough English BPE ratio
        long_docs = (est_tokens > 4096).sum()
        print(f"  Token estimate: median={est_tokens.median():.0f}, "
              f"95th pct={est_tokens.quantile(0.95):.0f}, "
              f"docs > 4096 tokens: {long_docs} ({100*long_docs/len(df):.1f}%)")
        if long_docs > 0:
            print(f"  Note: Long documents will need chunking before training. "
                  f"Consider using a sliding window with stride=512.")


validate_dataframe(
    df_pretrain, "pretrain",
    critical_cols=["text", "source", "domain", "split", "tier"]
)

# For SFT, validate the 'output' column
df_sft_validate = df_sft.copy()
df_sft_validate["text"] = df_sft_validate["output"]
validate_dataframe(
    df_sft_validate, "sft",
    critical_cols=["instruction", "output", "domain", "split"]
)

# Disclaimer presence check
disclaimer_present = df_sft["output"].str.contains("NALSA", case=False).sum()
print(f"\nSFT disclaimer check: {disclaimer_present}/{len(df_sft)} outputs contain NALSA reference.")

# SFT output deduplication
before_sft = len(df_sft)
df_sft = df_sft.drop_duplicates(subset=["output"])
print(f"SFT dedup: {before_sft} -> {len(df_sft)} (removed {before_sft - len(df_sft)} duplicate outputs)")

In [ ]:
# Export final Parquet files

pretrain_path = "data/final/pretrain_corpus.parquet"
sft_path = "data/final/sft_corpus.parquet"
card_path = "data/final/dataset_card.json"

df_pretrain.to_parquet(pretrain_path, index=False)
df_sft.to_parquet(sft_path, index=False)

# Dataset card for reproducibility
dataset_card = {
    "name": "PocketHarvey-Legal-Corpus-v1",
    "description": "Preprocessed Indian legal corpus for fine-tuning Bonsai-8B-mlx-1bit.",
    "splits": {
        "pretrain": {
            "path": pretrain_path,
            "rows": len(df_pretrain),
            "tier_distribution": df_pretrain["tier"].value_counts().to_dict(),
            "source_distribution": df_pretrain["source"].value_counts().to_dict(),
            "domain_distribution": df_pretrain["domain"].value_counts().to_dict(),
            "mixing_ratios": {"tier1": 0.50, "tier2": 0.35, "tier3": 0.15}
        },
        "sft": {
            "path": sft_path,
            "rows": len(df_sft),
            "format": "alpaca",
            "domain_distribution": df_sft["domain"].value_counts().to_dict(),
            "generation_method": "template_based_from_tier1_tier2_seed_docs"
        }
    },
    "pipeline_steps": [
        "raw_data_acquisition",
        "unicode_normalization_ftfy",
        "language_filtering_langdetect",
        "fineweb_style_quality_heuristics",
        "exact_deduplication_sha256",
        "near_dedup_minhash_lsh_jaccard_0.7",
        "educational_quality_scoring",
        "curriculum_tiering_3_levels",
        "synthetic_instruction_pair_generation",
        "smollm_nemotron_staged_mixing"
    ],
    "sources": [
        "joelniklaus/indian_legal_dataset",
        "law-ai/ildc_multi",
        "HuggingFaceFW/fineweb-edu (sampled, Indian legal filter)",
        "manually_authored_seed_texts"
    ],
    "licenses": {
        "joelniklaus/indian_legal_dataset": "MIT",
        "law-ai/ildc_multi": "CC BY 4.0",
        "HuggingFaceFW/fineweb-edu": "ODC-By",
        "seed_texts": "original_work_public_domain_law_basis"
    },
    "intended_use": "Continued pretraining and SFT of small language models for Indian legal literacy.",
    "not_intended_for": "Providing specific legal advice. All outputs must carry appropriate disclaimers."
}

with open(card_path, "w", encoding="utf-8") as f:
    json.dump(dataset_card, f, indent=2, ensure_ascii=False, default=str)

print("Export complete.")
print(f"  Pretrain corpus: {pretrain_path} ({len(df_pretrain)} rows)")
print(f"  SFT corpus:      {sft_path} ({len(df_sft)} rows)")
print(f"  Dataset card:    {card_path}")

## 10. Summary and Next Steps

### What this pipeline produced

Starting from raw, heterogeneous legal text across multiple sources, this pipeline outputs two clean, ready-to-train datasets:

1. **Pretraining corpus** (`pretrain_corpus.parquet`): Shuffled mix of Tier 1/2/3 documents in a 50/35/15 ratio. Each row has a `text` field and metadata. Feed this to a continued pretraining run on Bonsai-8B-1bit to specialize the model's prior distribution toward Indian legal text.

2. **SFT corpus** (`sft_corpus.parquet`): Instruction-response pairs in Alpaca format. Each response is grounded in a real source document and carries a standard NALSA disclaimer. Feed this to an SFT run after pretraining.

### Which techniques were used and why each was relevant

| Technique | Implementation | Relevance |
|-----------|---------------|----------|
| **Datatrove-style pipeline** | Modular filter stages mirroring datatrove's PipelineStep pattern | Reproducible, composable processing |
| **FineWeb heuristics** | Symbol ratio, bullet ratio, TTR, short line ratio | Removes low-quality web/OCR artifacts |
| **FineWebEdu quality signal** | Educational composite score (5 components) | Surfaces documents useful for SFT vs pretraining |
| **FinePDFs-style normalization** | `ftfy` + unicode NFC normalization | Cleans text extracted from government PDFs |
| **Exact deduplication** | SHA-256 hashing | Fast removal of exact copies |
| **MinHash LSH deduplication** | 128-perm MinHash, Jaccard threshold 0.7 | Removes near-duplicate court judgments |
| **SmolLM/Nemotron mixing** | 50/35/15 Tier 1/2/3 ratio | Prevents overfit to either raw judgments or easy text |
| **NeMo DataDesigner strategy** | Template-based instruction pair generation | Produces SFT data without a teacher LLM |

### What was intentionally excluded

- **Raw CommonCrawl WARC processing**: Requires distributed infrastructure (Spark or datatrove's `LocalPipelineExecutor` with many workers). The FineWebEdu sample already represents the best slice of CommonCrawl for our purposes. When scaling to 10B+ tokens, replace the FineWebEdu sampling in Section 2 with a full datatrove WARC pipeline.
- **LLM-based scoring (Bonsai as scorer)**: Loading Bonsai-8B in Colab just for scoring is wasteful at this data scale. The composite educational score is a reasonable proxy. At larger scale, replace with a trained educational quality classifier or use the perplexity of a 360M reference model.
- **LLM-based synthetic generation**: Replace the template-based generation in Section 7 with a `vllm` or `ollama` call to a local model to generate more diverse, contextually appropriate instruction pairs.

### Immediate next steps

1. Run continued pretraining on Bonsai-8B-mlx-1bit using `pretrain_corpus.parquet`. Recommended: 1 epoch, constant LR of 1e-5, batch size 8, context length 2048 (chunk long documents with stride 256).
2. Run SFT using `sft_corpus.parquet` with the Alpaca prompt template. Recommended: 3 epochs, LR 5e-6 with cosine decay.
3. Evaluate on InLegalBench or a manually constructed Indian legal QA test set.
4. Expand the seed texts to cover additional domains: land acquisition, SC/ST Atrocities Act, POCSO, RERA.
5. Add Hindi and Marathi seed texts — the current corpus is predominantly English.

### Downloading the outputs from Colab

Run the cell below to zip and download the final outputs.

In [ ]:
# Optional: Download outputs from Colab
# Uncomment and run if you want to download the Parquet files

# import shutil
# from google.colab import files
#
# shutil.make_archive("pocketharvey_corpus", "zip", "data/final")
# files.download("pocketharvey_corpus.zip")

# Quick sanity check — print sample rows from each split
print("=" * 60)
print("PRETRAIN CORPUS — sample row:")
print("=" * 60)
if len(df_pretrain) > 0:
    sample = df_pretrain.iloc[0]
    print(f"Source: {sample['source']}")
    print(f"Domain: {sample['domain']}")
    print(f"Tier: {sample['tier']}")
    print(f"Edu score: {sample['edu_score']:.3f}")
    print(f"Text (first 300 chars): {sample['text'][:300]}...")

print()
print("=" * 60)
print("SFT CORPUS — sample row:")
print("=" * 60)
if len(df_sft) > 0:
    sample = df_sft.iloc[0]
    print(f"Instruction: {sample['instruction']}")
    print(f"Domain: {sample['domain']}")
    print(f"Output (first 300 chars): {sample['output'][:300]}...")

In [ ]:
# Final statistics summary

print("PIPELINE SUMMARY")
print("-" * 50)
print(f"Raw documents ingested:          {len(raw_docs):>8,}")
print(f"After normalization filter:      {len(normalized_docs):>8,}")
print(f"After FineWeb quality heuristics:{len(quality_filtered):>8,}")
print(f"After exact deduplication:       {len(exact_deduped):>8,}")
print(f"After MinHash near-dedup:        {len(near_deduped):>8,}")
print(f"Pretrain corpus (final):         {len(df_pretrain):>8,}")
print(f"SFT pairs (final):               {len(df_sft):>8,}")
print("-" * 50)
print(f"Overall retention rate: {100*len(near_deduped)/max(len(raw_docs),1):.1f}%")